
# RFM-analüüs – UrbanStyle.ltd

**Meeskond:** Turundus

**Kuupäev:** 13.11.2026  

**Äriline väljakutse:** Marko soovib paremini mõista klientide ostukäitumist ja tuvastada väärtuslikumad kliendisegmendid.

**Eesmärk:** Analüüsida klientide ostude hiljutisust (Recency), ostusagedust (Frequency) ja ostude rahalist väärtust (Monetary), et toetada turundusotsuseid.


*----------------------------------------------------------------------------------------------------------------------------------------------------------------------*

**Roll A:** Data Loading — andmete laadimine ja liitmine

**Eesmärk:**
Laadida UrbanStyle.ltd müügi- ja kliendiandmed Supabase'ist Pandase DataFrame'idesse, kontrollida andmete struktuuri ja kvaliteeti ning ühendada tabelid.

In [1]:
import pandas as pd
import os 
from dotenv import load_dotenv
from supabase import create_client

# Laen ühenduse andmed .env failist
load_dotenv()

# Loon ühenduse Supabase'iga
supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

print("Supabase klient loodud")



Supabase klient loodud


In [2]:

# Funktsioon kõigi ridade laadimiseks Supabase'ist
def load_table(table_name):
    all_data = []
    page_size = 1000
    page = 0

    while True:
        response = (
            supabase.table(table_name)
            .select("*")
            .range(page * page_size, (page + 1) * page_size - 1)
            .execute()
        )

        all_data.extend(response.data)

        if len(response.data) < page_size:
            break

        page += 1

    return pd.DataFrame(all_data)


# Laadin müügiandmed
df_sales = load_table("sales")

# Kontrollin tulemust
print("Sales tabeli suurus:", df_sales.shape)
display(df_sales.head())


Sales tabeli suurus: (10118, 12)


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart


In [3]:
# Laadin kliendiandmed
df_customers = load_table("customers")

# Kontrollin tulemust
print("Customers tabeli suurus:", df_customers.shape)
display(df_customers.head())


Customers tabeli suurus: (3150, 9)


,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2637,Reet,Nurk,NaN,+372 5354 8280,Tallinn,2022-12-09,gold,1976
1,2723,Reet,Kuusik,NaN,+372 5603 8700,Tartu,2023-04-09,NaN,1998
2,2784,Mart,Pihl,NaN,+372 5536 5657,Tallinn,2022-10-30,gold,1989
3,3404,Maie,Tammik,NaN,+372 5291 9215,Tallinn,2020-03-26,NaN,2000
4,4278,Nele,Orav,NaN,+372 8700 9137,Tallinn,2024-05-10,NaN,1992


In [4]:
# Kontrollin tabelite struktuuri ja kvaliteeti (shape, dtypes, head(), puuduvad väärtused)
# SALES TABEL
print("=== SALES ===")
print("Suurus:", df_sales.shape)
print("\nAndmetüübid:")
print(df_sales.dtypes)
print("\nPuuduvad väärtused:")
print(df_sales.isna().sum())
display(df_sales.head())

# CUSTOMERS TABEL
print("\n=== CUSTOMERS ===")
print("Suurus:", df_customers.shape)
print("\nAndmetüübid:")
print(df_customers.dtypes)
print("\nPuuduvad väärtused:")
print(df_customers.isna().sum())
display(df_customers.head())


=== SALES ===
Suurus: (10118, 12)

Andmetüübid:
id                  int64
sale_id             int64
invoice_id            str
sale_date             str
customer_id       float64
product_id          int64
quantity            int64
unit_price        float64
total_price       float64
channel               str
store_location        str
payment_method        str
dtype: object

Puuduvad väärtused:
id                   0
sale_id              0
invoice_id           0
sale_date            0
customer_id        988
product_id           0
quantity             0
unit_price           0
total_price          0
channel              0
store_location    3462
payment_method       0
dtype: int64


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart



=== CUSTOMERS ===
Suurus: (3150, 9)

Andmetüübid:
customer_id          int64
first_name             str
last_name              str
email                  str
phone                  str
city                   str
registration_date      str
loyalty_tier           str
birth_year           int64
dtype: object

Puuduvad väärtused:
customer_id             0
first_name              0
last_name               0
email                 380
phone                   0
city                    0
registration_date       0
loyalty_tier         1260
birth_year              0
dtype: int64


,customer_id,first_name,last_name,email,phone,city,registration_date,loyalty_tier,birth_year
0,2637,Reet,Nurk,NaN,+372 5354 8280,Tallinn,2022-12-09,gold,1976
1,2723,Reet,Kuusik,NaN,+372 5603 8700,Tartu,2023-04-09,NaN,1998
2,2784,Mart,Pihl,NaN,+372 5536 5657,Tallinn,2022-10-30,gold,1989
3,3404,Maie,Tammik,NaN,+372 5291 9215,Tallinn,2020-03-26,NaN,2000
4,4278,Nele,Orav,NaN,+372 8700 9137,Tallinn,2024-05-10,NaN,1992


In [5]:

# Valin vajalikud kliendiandmed
customers_info = df_customers[
    ['customer_id', 'email', 'first_name']
]

# Ühendan müügi ja kliendid
df = pd.merge(
    df_sales,
    customers_info,
    on='customer_id',
    how='left',
    validate='many_to_one'
)

print("Enne ühendamist:", df_sales.shape)
print("Pärast ühendamist:", df.shape)

display(df.head())


Enne ühendamist: (10118, 12)
Pärast ühendamist: (10118, 14)


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,email,first_name
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,NaN,Hille
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,merle.luik@mail.ee,Merle
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,liina.saar@gmail.com,Liina
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,aili.pihl@yahoo.com,Aili
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,triin.lill@telia.ee,Triin


In [6]:

# Kontrollin ühendatud tabelit
print("Ühendatud tabeli suurus:", df.shape)

print("\nAndmetüübid:")
print(df.dtypes)

print("\nPuuduvad väärtused:")
print(df.isna().sum())

# Kontrollin nõutud veerge
required_columns = [
    'customer_id',
    'sale_date',
    'total_price',
    'email'
]

for column in required_columns:
    print(column, "olemas:", column in df.columns)

# Kontrollin, kas ridade arv säilis
print(
    "\nRidade arv säilis:",
    len(df) == len(df_sales)
)

display(df.head())


Ühendatud tabeli suurus: (10118, 14)

Andmetüübid:
id                  int64
sale_id             int64
invoice_id            str
sale_date             str
customer_id       float64
product_id          int64
quantity            int64
unit_price        float64
total_price       float64
channel               str
store_location        str
payment_method        str
email                 str
first_name            str
dtype: object

Puuduvad väärtused:
id                   0
sale_id              0
invoice_id           0
sale_date            0
customer_id        988
product_id           0
quantity             0
unit_price           0
total_price          0
channel              0
store_location    3462
payment_method       0
email             1944
first_name         988
dtype: int64
customer_id olemas: True
sale_date olemas: True
total_price olemas: True
email olemas: True

Ridade arv säilis: True


,id,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,email,first_name
0,1,1,INV-202301-00001,2023-01-10T00:00:00,2588.0,1274,2,234.79,469.58,pood,Tallinn,kaart,NaN,Hille
1,2,2,INV-202301-00002,2023-01-16T00:00:00,4338.0,1207,2,241.13,482.26,pood,Pärnu,järelmaks,merle.luik@mail.ee,Merle
2,3,3,INV-202301-00003,2023-01-05T00:00:00,4673.0,1264,1,258.46,221.19,pood,Pärnu,järelmaks,liina.saar@gmail.com,Liina
3,4,4,INV-202301-00004,2023-01-02T00:00:00,4677.0,1341,3,45.21,135.63,pood,Tartu,sularaha,aili.pihl@yahoo.com,Aili
4,5,5,INV-202301-00005,2023-01-13T00:00:00,2390.0,1284,1,99.57,99.57,pood,Tartu,kaart,triin.lill@telia.ee,Triin


In [7]:
# Kontrollin ühendatud tabeli kliendiandmeid
print("Puuduvad kliendi ID-d:", df["customer_id"].isna().sum())
print("Puuduvad e-postid:", df["email"].isna().sum())
print("Puuduvad eesnimed:", df["first_name"].isna().sum())

# Kontrollin, kas müügirida leidis vaste klienditabelist
vasteta = (
    df["customer_id"].notna()
    & df["first_name"].isna()
)

print("Vasteta kliendi ID-ga müügiridu:", vasteta.sum())

Puuduvad kliendi ID-d: 988
Puuduvad e-postid: 1944
Puuduvad eesnimed: 988
Vasteta kliendi ID-ga müügiridu: 0


**Roll A: Data Loading — kokkuvõte**

**Andmeallikad**
- sales: 10 118 rida, 12 veergu
- customers: 3150 rida, 9 veergu

**Kvaliteedikontroll**
- `sales.customer_id`: 988 puuduvat väärtust
- `sales.store_location`: 3462 puuduvat väärtust
- `customers.email`: 380 puuduvat väärtust
- `customers.loyalty_tier`: 1260 puuduvat väärtust
- `sale_date` ja `registration_date` on tekstivormingus

**Ühendamine**

Tabelid ühendati `customer_id` alusel (left merge).

Tulemuseks saadi 10 118 rida ja 14 veergu.
Kõik olemasoleva kliendi ID-ga müügiread leidsid klienditabelist vaste.

**Järeldus**

Andmed on edukalt Pandasesse laaditud ja ühendatud.
Supabase'i originaaltabeleid ei muudetud.
Edasine andmete puhastamine toimub roll B etapis.